### Pull Data 

In [14]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import torch.optim as optim

In [23]:
mill_data = pd.read_csv('Data Files/Site/Millcreek_obs.csv')
mill_bound = pd.read_csv('Data Files/Site/Millcreek_b.csv')
mill_river = pd.concat([mill_data,mill_bound],axis=1,join='outer',ignore_index=False)
mill_slope = pd.read_csv('Data Files/Site/Millcreek_slopefield.csv')

logan_data = pd.read_csv('Data Files/Site/Logan_obs.csv')
logan_bound = pd.read_csv('Data Files/Site/Logan_b.csv')
logan_river = pd.concat([logan_data,logan_bound],axis=1,join='outer',ignore_index=False)
logan_slope = pd.read_csv('Data Files/Site/Logan_slopefield.csv')

Build Chow ANN Values

In [36]:
#Clean straight (ndvi,m)
ndvi_c = torch.rand(20)*(0.2-0)+0
m_str = torch.rand(20)*(1-0)+0
cl_str = torch.stack((ndvi_c,m_str),dim=1)
n_clstr = torch.rand(20)*(0.033-0.025)+0.025

#vegetation straight (ndvi,m)
ndvi_veg = torch.rand(20)*(0.4-0.2)+0.2
m_str = torch.rand(20)*(1-0)+0
veg_str = torch.stack((ndvi_veg,m_str),dim=1)
n_vegstr = torch.rand(20)*(0.04-0.03)+0.03

#Clean windy (ndvi,m)
ndvi_c = torch.rand(20)*(0.2-0)+0
m_win = torch.rand(20)*(1.15-1)+1
cl_win = torch.stack((ndvi_c,m_win),dim=1)
n_clwin = torch.rand(20)*(0.045-0.033)+0.033

#vegetation windy (ndvi,m)
ndvi_veg = torch.rand(20)*(0.6-0.2)+0.2
m_win = torch.rand(20)*(1.15-1)+1
veg_win = torch.stack((ndvi_veg,m_win),dim=1)
n_vegwin = torch.rand(20)*(0.08-0.035)+0.035

#cobbles high sinuous
ndvi_roc = torch.rand(20)*(0.8-0.4)+0.4
m_hi = torch.rand(20)*(1.3-1.15)+1.15
roc_hi = torch.stack((ndvi_roc, m_hi), dim=1)
n_rchi = torch.rand(20)*(0.08-0.05)+0.05

train = torch.cat([cl_str,veg_str,cl_win,veg_win,roc_hi], dim=0)

rough_true = torch.cat([
    0.028 + 0.002*torch.randn(20),
    0.035 + 0.002*torch.randn(20),
    0.039 + 0.003*torch.randn(20),
    0.060 + 0.004*torch.randn(20),
    0.065 + 0.004*torch.randn(20),
], dim=0).unsqueeze(1)
rough_true=rough_true.unsqueeze(1)

In [18]:
#mannings ANN

class Mannings_ANN(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 16),   # NDVI + sinuosity
            nn.ReLU(),

            nn.Linear(16, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

        self.n_min = 0.025
        self.n_max = 0.080

    def forward(self, features):
        z = self.net(features)
        z = torch.sigmoid(z)
        n = self.n_min + (self.n_max - self.n_min) * z
        return n

In [37]:
roughness = Mannings_ANN()

loss_func = nn.MSELoss()
optimize = optim.Adam(roughness.parameters(),lr=0.001)
phase = "pretrain_n" #phases to couple PINN and ANN

for epoch in range(500):
    #smooth-ANN turn on 
    alpha = torch.tensor(
            1 / (1 + np.exp(-0.01 * (epoch - 1000))),
            dtype=torch.float32
        )

    # forward pass
    predict_n = roughness(torch.tensor(train, dtype=torch.float32))
    loss_data = torch.mean((predict_n - rough_true)**2)
    loss_smooth = torch.mean((predict_n[1:] - predict_n[:-1])**2) #because n is a function of x here

    #loss
    loss = loss_data + 0.01 * loss_smooth
    optimize.zero_grad()

    # backpropagation
    loss.backward()
    optimize.step()

    # print progress
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")

/var/folders/gt/m7_lxk5j1mn90fqjrx8y57300000gn/T/ipykernel_15835/1785700336.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  predict_n = roughness(torch.tensor(train, dtype=torch.float32))


Epoch 0, Loss: 0.000290
Epoch 100, Loss: 0.000209
Epoch 200, Loss: 0.000208
Epoch 300, Loss: 0.000208
Epoch 400, Loss: 0.000208


In [38]:
test_input = torch.tensor([[0.5, 1]])  # should be vegetation windy 
predict_n = roughness(test_input)

print("\nPredicted Manning's n:", predict_n.item())


Predicted Manning's n: 0.045027852058410645
